# 💊 HỆ THỐNG NHẬN DIỆN THUỐC VÀ ĐỐI SOÁT TƯƠNG TÁC DƯỢC LÝ
### Multiple Pill Recognition & Clinical Drug-Drug Interaction Safety Platform
---
Notebook này tự động thiết lập và chạy toàn bộ hệ thống gồm:
1. **Mô hình AI Computer Vision**: YOLOv11 Segmentation + ResNet-18 Attribute Multi-Head + PaddleOCR v3
2. **Cơ sở dữ liệu Dược thư Quốc gia (RxNorm CSDL)**: Định danh thuốc, mã khắc, hoạt chất, ma trận DDI
3. **Giao diện Lâm sàng Web & Mobile**: Streamlit + Cloudflare Tunnel (Truy cập trực tiếp không cần mật khẩu)
---
## 🔹 BƯỚC 1: Clone Mã Nguồn (Nhánh mergerBe/Fe) & Cài Đặt PaddleOCR v3 Chuẩn Hóa

In [ ]:
import os
import shutil

# 1. Luôn quay về thư mục an toàn trước khi xóa repo cũ
%cd /kaggle/working
if os.path.exists('/kaggle/working/repo'):
    shutil.rmtree('/kaggle/working/repo')

# 2. Clone nhánh mergerBe/Fe chuẩn với shallow clone tốc độ cao
!git clone --depth 1 -b mergerBe/Fe https://github.com/GOx9-P/Multiple-Pill-Recognition-And-Interaction-Safety.git /kaggle/working/repo

# 3. Chuyển vào thư mục repo
%cd /kaggle/working/repo

# 4. Gỡ các package gây xung đột cũ
!pip uninstall -y -q paddlepaddle paddlepaddle-gpu paddleocr paddlex

# 5. Cài đặt các thư viện từ requirements.txt
!pip install -q --no-cache-dir -r requirements.txt

# 6. Cài đặt Paddle GPU 3.0.0 & PaddleOCR v3.0.3 chính hãng từ official cu118 index theo chuẩn của đồng đội
!pip install -q paddlepaddle-gpu==3.0.0 --index-url https://www.paddlepaddle.org.cn/packages/stable/cu118/
!pip install -q paddleocr==3.0.3 paddlex==3.0.3 "numpy==1.26.4" "opencv-python-headless==4.10.0.84"

print("✅ BƯỚC 1 HOÀN TẤT: Đã clone mã nguồn và cài đặt toàn bộ package thành công!")

---
## 🔹 BƯỚC 2: Tự Động Tải 3 Datasets Từ Kaggle & Nạp Weights

In [ ]:
import os
import shutil
import glob
from pathlib import Path
import kagglehub

os.chdir('/kaggle/working/repo')
repo_root = Path('/kaggle/working/repo')
seg_dir = repo_root / 'models/segmentation_yolov11_full_finetune'
attr_dir = repo_root / 'models/attribute_resnet18_last_blocks_finetune'
db_seed_dir = repo_root / 'database_seed'

seg_dir.mkdir(parents=True, exist_ok=True)
attr_dir.mkdir(parents=True, exist_ok=True)
db_seed_dir.mkdir(parents=True, exist_ok=True)

print("🚀 Đang tự động tải 3 Datasets trực tiếp từ link Kaggle...")

# 1. Tải YOLOv11 Segmentation Model
print("1/3. Đang tải mô hình YOLOv11 Segmentation...")
try:
    seg_download_path = kagglehub.dataset_download('nnphuchcmus/pill-segmentation-model')
    for f in glob.glob(f'{seg_download_path}/**/*.pt', recursive=True):
        target_pt = seg_dir / 'yolov11m_seg_mediseg_full_finetune_v1.pt'
        shutil.copy(f, target_pt)
        size_mb = os.path.getsize(target_pt) / (1024 * 1024)
        print(f"  ✓ Đã nạp YOLOv11-Seg: {target_pt.name} ({size_mb:.1f} MB)")
        break
except Exception as e:
    print(f"  ⚠️ Tải hub lỗi ({e}), đang quét trong /kaggle/input...")
    for f in glob.glob('/kaggle/input/**/*.pt', recursive=True):
        if 'seg' in f.lower() or 'yolo' in f.lower():
            shutil.copy(f, seg_dir / 'yolov11m_seg_mediseg_full_finetune_v1.pt')
            print(f"  ✓ Đã nạp từ input: {f}")
            break

# 2. Tải ResNet-18 Attribute Recognition Artifacts
print("2/3. Đang tải mô hình ResNet-18 Attribute...")
try:
    attr_download_path = kagglehub.dataset_download('nnphuchcmus/attrubute-artifact')
    copied_attr = 0
    for f in glob.glob(f'{attr_download_path}/**/*', recursive=True):
        if os.path.isfile(f):
            dest = attr_dir / os.path.basename(f)
            shutil.copy(f, dest)
            copied_attr += 1
    print(f"  ✓ Đã nạp {copied_attr} files cấu hình & weights cho ResNet-18!")
except Exception as e:
    print(f"  ⚠️ Tải hub lỗi ({e}), đang quét trong /kaggle/input...")
    for f in glob.glob('/kaggle/input/**/*', recursive=True):
        if os.path.isfile(f):
            bname = os.path.basename(f).lower()
            if 'best' in bname and bname.endswith('.pt'):
                shutil.copy(f, attr_dir / 'best.pt')
            elif 'mapping' in bname and bname.endswith('.json'):
                shutil.copy(f, attr_dir / 'label_mapping.json')
            elif 'threshold' in bname and bname.endswith('.json'):
                shutil.copy(f, attr_dir / 'optimal_thresholds.json')
            elif 'model_config' in bname or ('config' in bname and bname.endswith('.yaml')):
                shutil.copy(f, attr_dir / 'model_config.yaml')

# 3. Tải Cơ sở Dữ liệu Dược thư Quốc gia (Database Seeds)
print("3/3. Đang tải CSDL Dược thư Quốc gia...")
try:
    db_download_path = kagglehub.dataset_download('nnphuchcmus/pill-safety-database')
    db_count = 0
    for f in glob.glob(f'{db_download_path}/**/*.json', recursive=True):
        dest = db_seed_dir / os.path.basename(f)
        shutil.copy(f, dest)
        db_count += 1
    print(f"  ✓ Đã đồng bộ {db_count} datasets dược thư JSON!")
except Exception as e:
    print(f"  ⚠️ Tải hub lỗi ({e}), đang dùng seed mặc định của repo.")

# 4. Cấu hình file .env cho SQLite
with open(repo_root / '.env', 'w', encoding='utf-8') as f:
    f.write('DATABASE_URL=sqlite:///./medication.db\n')
    f.write('LLM_PROVIDER=fallback\n')

# 5. Bảng kiểm tra xác thực sự tồn tại của các file model
print("\n" + "=" * 65)
print("📋 BẢNG KIỂM TRA MODEL ARTIFACTS TRÊN MÔI TRƯỜNG:")
check_files = [
    seg_dir / 'yolov11m_seg_mediseg_full_finetune_v1.pt',
    attr_dir / 'best.pt',
    attr_dir / 'label_mapping.json',
    attr_dir / 'optimal_thresholds.json',
    attr_dir / 'model_config.yaml',
]
all_ready = True
for cf in check_files:
    if cf.exists():
        size = os.path.getsize(cf)
        size_str = f"{size/(1024*1024):.1f} MB" if size > 1024*1024 else f"{size/1024:.1f} KB"
        print(f"  [OK] {cf.name:<32} ({size_str})")
    else:
        print(f"  [MISSING] {cf.name:<32}")
        all_ready = False
print("=" * 65)
if all_ready:
    print("🎉 BƯỚC 2 HOÀN TẤT 100%: Toàn bộ Model AI đã sẵn sàng trên đĩa!")
else:
    print("⚠️ Cảnh báo: Một số file chưa tìm thấy, vui lòng kiểm tra lại log tải ở trên.")

---
## 🔹 BƯỚC 3: Khởi Tạo & Nạp CSDL Dược Thư (SQLite Database)

In [ ]:
import sys
import os

os.chdir('/kaggle/working/repo')
src_path = '/kaggle/working/repo/src'
if src_path not in sys.path:
    sys.path.insert(0, src_path)
os.environ['PYTHONPATH'] = f"{src_path}:{os.environ.get('PYTHONPATH', '')}"

# Chạy seed database
!PYTHONPATH=src python -m pill_safety.database.scripts.seed

from pill_safety.database.session import SessionLocal
from pill_safety.database.models import DrugProduct, DrugInteraction

with SessionLocal() as db:
    total_drugs = db.query(DrugProduct).count()
    total_ddi = db.query(DrugInteraction).count()
    print("=" * 65)
    print(f"📊 CSDL ĐÃ NẠP THÀNH CÔNG: {total_drugs} sản phẩm thuốc | {total_ddi} cặp tương tác DDI")
    print("=" * 65)

print("✅ BƯỚC 3 HOÀN TẤT 100%!")

---
## 🔹 BƯỚC 4: Khởi Chạy Web & Mobile UI (Cloudflare Tunnel)

In [ ]:
import subprocess
import time
import os

os.chdir('/kaggle/working/repo')

# 1. Tắt các tiến trình Streamlit cũ nếu có
!fuser -k 8501/tcp 2>/dev/null || true

# 2. Tải và cài đặt Cloudflared (nếu chưa có)
!which cloudflared > /dev/null 2>&1 || (wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1)

# 3. Khởi chạy Streamlit ở background (tắt fileWatcher để không hiện log torch.classes)
cmd_streamlit = "streamlit run app.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false --server.enableWebsocketCompression false --server.fileWatcherType none"
subprocess.Popen(cmd_streamlit, shell=True)
time.sleep(3)

print("🚀 Streamlit đã khởi chạy thành công!")
print("🌐 Đang mở đường link Cloudflare Public Tunnel...")
print("👉 Click trực tiếp vào đường link https://*.trycloudflare.com bên dưới để mở ứng dụng:\n")

# 4. Mở tunnel Cloudflare trực tiếp
!cloudflared tunnel --url http://localhost:8501